In [61]:
!pip install torch torchvision torchaudio
!pip install torchcrf tqdm scikit-learn

In [62]:
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from tqdm import tqdm, trange
from TorchCRF import CRF

In [63]:
with open("genia/genia_train_dev.json") as f:
    data = json.load(f)

train_data, dev_data = train_test_split(
    data,
    test_size=0.1,
    random_state=42
)

with open("genia/vocab.json") as f:
    vocab = json.load(f)

embedding_matrix = np.load("genia/vocab_embed_bio.npy")

In [64]:
def tokens_to_ids(tokens, vocab):
    return [vocab.get(token.lower(), vocab["<unk>"]) for token in tokens]

In [65]:
entity_types = ["DNA", "RNA", "cell_type", "protein", "cell_line"]

def build_bio_label_map(entity_types):
    label2id = {"O": 0}
    idx = 1
    for ent in entity_types:
        label2id[f"B-{ent}"] = idx
        idx += 1
        label2id[f"I-{ent}"] = idx
        idx += 1
    return label2id

label2id = build_bio_label_map(entity_types)
id2label = {v: k for k, v in label2id.items()}

In [66]:
def spans_to_bio_levels(tokens, entities, max_levels=2):
    seq_len = len(tokens)
    levels = [["O"] * seq_len for _ in range(max_levels)]

    entities = sorted(entities, key=lambda x: (x["end"] - x["start"]), reverse=True)
    level_ends = [-1] * max_levels

    for ent in entities:
        start, end, label = ent["start"], ent["end"], ent["type"]

        # 🔥 safety check
        if start >= seq_len or end > seq_len:
            continue

        for lvl in range(max_levels):
            if start >= level_ends[lvl]:
                levels[lvl][start] = f"B-{label}"

                for i in range(start + 1, end):   # 🔥 FIXED
                    levels[lvl][i] = f"I-{label}"

                level_ends[lvl] = end
                break

    return levels







def bio_to_spans(tags, id2label):
    spans = []
    start = None
    label = None

    for i, tag_id in enumerate(tags):
        tag = id2label[tag_id]

        if tag.startswith("B-"):
            if start is not None:
                spans.append((start, i, label))
            start = i
            label = tag[2:]

        elif tag.startswith("I-"):
            continue

        else:  # "O"
            if start is not None:
                spans.append((start, i, label))
                start = None

    if start is not None:
        spans.append((start, len(tags), label))

    return spans

In [67]:
class GeniaCRFDataset(torch.utils.data.Dataset):
    def __init__(self, data, vocab, label2id, max_levels=2):
        self.data = data
        self.vocab = vocab
        self.label2id = label2id
        self.max_levels = max_levels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = item["tokens"]
        entities = item["entities"]

        token_ids = tokens_to_ids(tokens, self.vocab)
        levels = spans_to_bio_levels(tokens, entities, self.max_levels)

        label_levels = []
        for lvl in levels:
            label_levels.append([self.label2id.get(tag, 0) for tag in lvl])

        return {
            "token_ids": torch.tensor(token_ids),
            "labels": [torch.tensor(l) for l in label_levels]
        }

In [68]:
def collate_fn_crf(batch):
    token_ids = [item["token_ids"] for item in batch]
    labels_levels = [item["labels"] for item in batch]

    padded_tokens = pad_sequence(token_ids, batch_first=True, padding_value=0)

    num_levels = len(labels_levels[0])
    padded_labels = []

    for lvl in range(num_levels):
        lvl_labels = [labels_levels[i][lvl] for i in range(len(batch))]
        padded_lvl = pad_sequence(lvl_labels, batch_first=True, padding_value=0)
        padded_labels.append(padded_lvl)

    return {
        "token_ids": padded_tokens,
        "labels": padded_labels
    }

In [69]:
class EmbeddingLayer(nn.Module):
    def __init__(self, embedding_matrix):
        super().__init__()
        num_embeddings, emb_dim = embedding_matrix.shape
        self.embedding = nn.Embedding(num_embeddings, emb_dim)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))
        self.embedding.weight.requires_grad = True

    def forward(self, token_ids):
        return self.embedding(token_ids)

In [70]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim // 2,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        outputs, _ = self.lstm(x)
        return outputs

In [71]:
class BiLSTM_MultiCRF(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, num_labels, num_levels=2):
        super().__init__()

        emb_dim = embedding_matrix.shape[1]

        self.embedding = EmbeddingLayer(embedding_matrix)
        self.encoder = Encoder(emb_dim, hidden_dim)

        self.hidden2tag = nn.ModuleList([
            nn.Linear(hidden_dim, num_labels) for _ in range(num_levels)
        ])

        self.crfs = nn.ModuleList([
            CRF(num_labels) for _ in range(num_levels)
        ])

    def forward(self, token_ids, labels=None, mask=None):
        x = self.embedding(token_ids)
        H = self.encoder(x)

        losses = []
        outputs = []

        for i in range(len(self.crfs)):
            emissions = self.hidden2tag[i](H)        # (B, T, C)
            emissions = emissions.permute(1, 0, 2)   # (T, B, C)

            mask_ = mask.permute(1, 0)               # (T, B)

            if labels is not None:
                tags = labels[i].permute(1, 0)       # (T, B)

                loss = -self.crfs[i](emissions, tags, mask=mask_)
                loss = loss.mean()                  # 🔥 FIX HERE

                losses.append(loss)

            decoded = self.crfs[i].viterbi_decode(emissions, mask=mask_)
            outputs.append(decoded)

        if labels is not None:
            return sum(losses), outputs

        return outputs

In [72]:
train_dataset = GeniaCRFDataset(train_data, vocab, label2id)
dev_dataset = GeniaCRFDataset(dev_data, vocab, label2id)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn_crf)
dev_loader = DataLoader(dev_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn_crf)

In [73]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTM_MultiCRF(
    embedding_matrix=embedding_matrix,
    hidden_dim=256,
    num_labels=len(label2id),
    num_levels=2
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [74]:
def train_epoch_crf(model, dataloader):
    model.train()
    total_loss = 0

    for batch in tqdm(dataloader, desc="Training", leave=False):
        optimizer.zero_grad()

        token_ids = batch["token_ids"].to(device)
        labels = [l.to(device) for l in batch["labels"]]

        mask = (token_ids != 0)

        loss, _ = model(token_ids, labels, mask)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [75]:
def compute_f1(all_preds, all_labels):
    tp, fp, fn = 0, 0, 0

    for pred, true in zip(all_preds, all_labels):
        if pred != 0 and pred == true:
            tp += 1
        elif pred != 0 and pred != true:
            fp += 1
        elif true != 0 and pred != true:
            fn += 1

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1





def compute_span_f1(pred_spans, true_spans):
    pred_set = set(pred_spans)
    true_set = set(true_spans)

    tp = len(pred_set & true_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1

In [76]:
def get_span_predictions_crf(model, dataloader, id2label):
    model.eval()

    all_pred_spans = []
    all_true_spans = []

    with torch.no_grad():
        for batch in dataloader:
            token_ids = batch["token_ids"].to(device)
            labels = [l.to(device) for l in batch["labels"]]

            mask = (token_ids != 0)
            outputs = model(token_ids, mask=mask)

            B = token_ids.shape[0]

            for b in range(B):
                for lvl in range(len(outputs)):

                    pred_seq = outputs[lvl][b]
                    true_seq = labels[lvl][b].cpu().tolist()

                    # remove padding
                    length = mask[b].sum().item()
                    pred_seq = pred_seq[:length]
                    true_seq = true_seq[:length]

                    pred_spans = bio_to_spans(pred_seq, id2label)
                    true_spans = bio_to_spans(true_seq, id2label)

                    all_pred_spans.extend(pred_spans)
                    all_true_spans.extend(true_spans)

    return all_pred_spans, all_true_spans

In [ ]:
EPOCHS = 5

for epoch in trange(EPOCHS, desc="Epochs"):
    train_loss = train_epoch_crf(model, train_loader)

    pred_spans, true_spans = get_span_predictions_crf(model, dev_loader, id2label)
    p, r, f1 = compute_span_f1(pred_spans, true_spans)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Precision: {p:.4f}, Recall: {r:.4f}, F1: {f1:.4f}")
    print("------")

Epochs:  20%|██        | 1/5 [04:05<16:22, 245.69s/it]


Epoch 1
Train Loss: -208.4080
Precision: 0.8696, Recall: 0.0274, F1: 0.0532
------


Epochs:  40%|████      | 2/5 [07:59<11:56, 238.99s/it]


Epoch 2
Train Loss: -665.9414
Precision: 0.8750, Recall: 0.0288, F1: 0.0558
------


Epochs:  60%|██████    | 3/5 [11:55<07:54, 237.24s/it]


Epoch 3
Train Loss: -1156.8509
Precision: 0.8500, Recall: 0.0233, F1: 0.0454
------


Epochs:  80%|████████  | 4/5 [15:52<03:57, 237.31s/it]


Epoch 4
Train Loss: -1647.4343
Precision: 0.8400, Recall: 0.0288, F1: 0.0557
------


In [ ]:
EPOCHS = 5
best_f1 = 0

for epoch in trange(EPOCHS, desc="Epochs"):
    train_loss = train_epoch_crf(model, train_loader)

    pred_spans, true_spans = get_span_predictions_crf(model, dev_loader, id2label)
    p, r, f1 = compute_span_f1(pred_spans, true_spans)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Precision: {p:.4f}, Recall: {r:.4f}, F1: {f1:.4f}")
    print("------")

    # 🔥 SAVE BEST MODEL
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "bilstm_crf_best.pt")
        print("✅ Best model saved!")

In [ ]:
model.load_state_dict(torch.load("bilstm_crf_best.pt"))
model.eval()